# Day3_11 · 여러 집단의 차이 비교

네 가지 건물 방향의 냉방 부하를 그림, ANOVA, 사후비교와 효과크기로 확인합니다.

## 이 노트북에서 확인할 내용

- 세 개 이상 집단의 요약표와 분포를 비교할 수 있습니다.
- 전체 비교와 집단 쌍 비교를 구분할 수 있습니다.
- p-value와 효과크기를 함께 기록할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-11-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-11-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-11-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-11-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-11-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-11-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-11-07 · 건물 에너지자료 읽기

In [ ]:
building = pd.read_csv(DATA_DIR / 'building_energy.csv')

In [ ]:
building.head()

### D3B-11-08 · 건물 방향 종류 확인

In [ ]:
building['orientation'].value_counts().sort_index()

### D3B-11-09 · 방향별 냉방 부하 요약

In [ ]:
orientation_summary = building.groupby('orientation')['cooling_load'].agg(['count', 'mean', 'median', 'std'])

In [ ]:
orientation_summary.round(2)

### D3B-11-10 · 방향별 냉방 부하 상자그림

In [ ]:
sns.boxplot(data=building, x='orientation', y='cooling_load')
sns.stripplot(data=building, x='orientation', y='cooling_load', color='black', alpha=0.15, size=2)
plt.xlabel('Orientation code')
plt.ylabel('Cooling load')
plt.show()

## 여러 집단을 한 번에 비교하기

먼저 네 집단의 평균이 모두 같다는 가정을 놓고, 관찰한 집단 사이 차이가 집단 안의 흩어짐에 비해 큰지 확인합니다.

### D3B-11-11 · 여러 집단 비교 함수 불러오기

In [ ]:
from scipy.stats import f_oneway, kruskal, tukey_hsd

### D3B-11-12 · 방향별 수치 목록 만들기

In [ ]:
orientation_groups = [group['cooling_load'].to_numpy() for _, group in building.groupby('orientation')]

In [ ]:
pd.Series([len(group) for group in orientation_groups], index=sorted(building['orientation'].unique()), name='rows')

### D3B-11-13 · ANOVA 계산

In [ ]:
anova_result = f_oneway(*orientation_groups)

In [ ]:
anova_result

### D3B-11-14 · 순위를 이용한 여러 집단 비교

In [ ]:
kruskal_result = kruskal(*orientation_groups)

In [ ]:
kruskal_result

### D3B-11-15 · 두 결과를 한 표로 정리

In [ ]:
overall_test_summary = pd.DataFrame({
    'method': ['one-way ANOVA', 'Kruskal-Wallis'],
    'statistic': [anova_result.statistic, kruskal_result.statistic],
    'p_value': [anova_result.pvalue, kruskal_result.pvalue],
})

In [ ]:
overall_test_summary

전체 검정의 p-value가 작더라도 어느 방향끼리 다른지는 아직 알 수 없습니다. 다음 표에서 방향 쌍을 따로 확인합니다.

### D3B-11-16 · Tukey 사후비교 계산

In [ ]:
tukey_result = tukey_hsd(*orientation_groups)

In [ ]:
tukey_result

### D3B-11-17 · 방향 쌍의 p-value 표 만들기

In [ ]:
orientation_names = sorted(building['orientation'].unique())
tukey_pvalues = pd.DataFrame(tukey_result.pvalue, index=orientation_names, columns=orientation_names)

In [ ]:
tukey_pvalues.round(4)

### D3B-11-18 · 방향 쌍의 p-value를 색으로 보기

In [ ]:
sns.heatmap(tukey_pvalues, annot=True, fmt='.3f', cmap='viridis_r', vmin=0, vmax=0.1)
plt.xlabel('Orientation code')
plt.ylabel('Orientation code')
plt.title('Tukey pairwise p-values')
plt.show()

## 집단이 설명하는 차이의 크기 확인하기

### D3B-11-19 · 전체 냉방 부하 평균 계산

In [ ]:
grand_mean = building['cooling_load'].mean()

In [ ]:
grand_mean

### D3B-11-20 · 집단 사이 제곱합 계산

In [ ]:
between_sum_squares = sum(len(group) * (group.mean() - grand_mean) ** 2 for group in orientation_groups)

In [ ]:
between_sum_squares

### D3B-11-21 · 전체 제곱합 계산

In [ ]:
total_sum_squares = ((building['cooling_load'] - grand_mean) ** 2).sum()

In [ ]:
total_sum_squares

### D3B-11-22 · 방향 집단이 설명하는 비율 계산

In [ ]:
eta_squared = between_sum_squares / total_sum_squares

In [ ]:
eta_squared

이 비율을 eta squared라고 부릅니다. 현재 자료에서 방향 집단 사이 차이가 전체 냉방 부하 변동의 어느 정도를 차지하는지 요약하지만, 방향의 인과효과를 증명하지는 않습니다.

### D3B-11-23 · 비교할 결과 열 정하기

In [ ]:
# cooling_load를 heating_load로 바꾸어 보세요.
outcome_column = 'heating_load'

In [ ]:
outcome_column

### D3B-11-24 · 선택한 결과의 방향별 요약

In [ ]:
chosen_orientation_summary = building.groupby('orientation')[outcome_column].agg(['count', 'mean', 'median', 'std'])

In [ ]:
chosen_orientation_summary.round(2)

### D3B-11-25 · 선택한 결과의 방향별 그림

In [ ]:
sns.boxplot(data=building, x='orientation', y=outcome_column)
plt.title(f'Orientation and {outcome_column}')
plt.show()

### D3B-11-26 · 해석 순서 정리

In [ ]:
anova_note = pd.Series({
    '먼저 볼 것': '방향별 관측 수와 상자그림',
    '전체 질문': '네 집단 중 적어도 하나가 다른가?',
    '후속 질문': '어느 방향 쌍에서 차이가 나타나는가?',
    '함께 기록': '효과크기와 자료 설계의 한계',
})

In [ ]:
anova_note

### D3B-11-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''building DataFrame을 사용하세요. 기존 셀을 수정하지 말고 orientation별 heating_load를 비교하는 표와 상자그림을 새 셀로 제안해 주세요. 그 다음 ANOVA가 답하는 질문과 Tukey 비교가 답하는 질문을 각각 한 문장으로 적어 주세요. 원인으로 단정하지 마세요.'''.strip()

In [ ]:
codex_request

### D3B-11-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['orientation을 범주로 다루는가?', '전체 비교와 쌍별 비교를 구분하는가?', '효과크기와 그림도 함께 보는가?'], name='확인할 내용')

In [ ]:
codex_checklist